# GeoPython 2026 — Día 1c: Introducción a Pandas

**Curso de Análisis Espacial con Python**  
Gabinete de Formación del CSIC — Estación Biológica de Doñana, Sevilla

---

> **Por qué pandas antes de GeoPandas:** GeoPandas es esencialmente pandas con una
> columna extra de geometría. Dominar las operaciones básicas de pandas aquí nos
> permitirá centrarnos en la parte geoespacial en el Día 2.

## Contenido

1. Series y DataFrame — los dos objetos fundamentales
2. Crear DataFrames
3. Explorar datos (`head`, `info`, `describe`)
4. Selección e indexación (`loc`, `iloc`, filtros booleanos)
5. Añadir, modificar y eliminar columnas
6. Datos ausentes
7. Agrupaciones y agregaciones (`groupby`)
8. Combinar DataFrames (`merge`, `concat`)
9. Leer y escribir archivos (CSV, Excel)
10. Visualización rápida con pandas
11. Conexión con GeoPandas

---
## 1. Series y DataFrame

Pandas tiene dos estructuras de datos principales:

| Estructura | Analogía | Descripción |
|------------|----------|-------------|
| `Series`   | columna de Excel | Array 1D con índice etiquetado |
| `DataFrame`| hoja de Excel | Tabla 2D, colección de Series |

```
DataFrame
  ├── columna_1  →  Series
  ├── columna_2  →  Series
  └── columna_3  →  Series
```

---
## El ecosistema científico de Python

Antes de entrar en pandas conviene tener una visión de conjunto de las librerías
que forman el **stack científico de Python**. Todas ellas se construyen unas sobre otras:

```
                        Tu análisis / aplicación
                               ↑
         GeoPandas · Rasterio · Shapely · scikit-learn · …
                               ↑
              pandas ──────────┤
              matplotlib       │   (construidas sobre NumPy)
              SciPy ───────────┘
                               ↑
                            NumPy
                               ↑
                     Python + C / Fortran
```

---

### NumPy

**NumPy** (Numerical Python) es la base de casi todo el stack científico. Proporciona:

- El objeto `ndarray`: array N-dimensional **homogéneo** (todos los elementos del mismo tipo),
  almacenado en memoria contigua → operaciones vectorizadas muy rápidas sin bucles Python
- Álgebra lineal, transformadas de Fourier, números aleatorios
- Broadcasting: operar arrays de distintas formas sin copiar datos

```python
import numpy as np

a = np.array([1, 2, 3, 4, 5])
print(a * 2)          # [2 4 6 8 10]  — sin bucle
print(a.mean())       # 3.0
print(a[a > 2])       # [3 4 5]       — indexación booleana

m = np.zeros((3, 4))        # matriz 3×4 de ceros
m = np.random.rand(100, 3)  # 100 filas, 3 columnas de valores aleatorios [0,1)
```

> **Por qué es tan rápido:** los bucles se ejecutan en C/Fortran compilado, no en
> el intérprete de Python. Trabajar con arrays de NumPy en lugar de listas Python
> puede suponer mejoras de 10×–100× en velocidad.

📖 https://numpy.org/doc/stable/user/quickstart.html

---

### SciPy

**SciPy** amplía NumPy con algoritmos científicos de alto nivel, organizados en submódulos:

| Submódulo | Contenido |
|-----------|-----------|
| `scipy.stats` | Distribuciones estadísticas, tests (t-test, chi², KS…), correlaciones |
| `scipy.spatial` | KD-trees, triangulación Delaunay, distancias espaciales |
| `scipy.interpolate` | Interpolación 1D/2D (splines, griddata) |
| `scipy.optimize` | Minimización, ajuste de curvas, raíces |
| `scipy.signal` | Filtros, convolución, análisis espectral |
| `scipy.ndimage` | Procesado de imágenes N-dimensionales |
| `scipy.linalg` | Álgebra lineal extendida (SVD, descomposiciones…) |

```python
from scipy import stats

# Test de normalidad
stat, p = stats.shapiro([2.1, 3.4, 2.8, 3.1, 2.9])
print(f'p-value: {p:.3f}')   # si p > 0.05 no se rechaza normalidad

# Correlación de Pearson
r, p = stats.pearsonr([1, 2, 3, 4], [1.1, 1.9, 3.2, 3.8])
print(f'r = {r:.3f}')
```

📖 https://docs.scipy.org/doc/scipy/

---

### El ecosistema completo (selección)

| Librería | Para qué sirve | Sobre qué se apoya |
|----------|---------------|--------------------|
| **NumPy** | Arrays numéricos, operaciones vectorizadas | C/Fortran |
| **SciPy** | Algoritmos científicos (estadística, optimización…) | NumPy |
| **pandas** | Tablas de datos (DataFrames), I/O, análisis | NumPy |
| **Matplotlib** | Gráficas 2D (la base de la visualización Python) | NumPy |
| **Seaborn** | Gráficas estadísticas de alto nivel | Matplotlib + pandas |
| **scikit-learn** | Machine learning | NumPy + SciPy |
| **Shapely** | Geometrías vectoriales (puntos, líneas, polígonos) | GEOS (C++) |
| **Fiona** | Lectura/escritura de formatos vectoriales (SHP, GeoJSON…) | GDAL |
| **GeoPandas** | DataFrames con geometría, operaciones espaciales | pandas + Shapely + Fiona |
| **Rasterio** | Lectura/escritura y análisis de datos raster | GDAL |
| **Xarray** | Arrays N-dimensionales etiquetados (NetCDF, datos climáticos) | NumPy + pandas |
| **Folium / ipyleaflet** | Mapas interactivos en el notebook | Leaflet.js |

> En este curso usaremos principalmente **pandas → GeoPandas → Rasterio**.  
> Conocer NumPy y SciPy nos ayuda a entender cómo funcionan por dentro y a hacer
> análisis más potentes cuando los necesitemos.

In [ ]:
import pandas as pd
import numpy as np

# Series: array 1D con índice
altitudes = pd.Series(
    [828, 4478, 2382, 1085, 2643],
    index=['Burj Khalifa', 'Mont Blanc', 'Teide', 'Sierra Nevada', 'Mulhacén'],
    name='altitud_m'
)
print(altitudes)
print(f'\nTipo: {type(altitudes)}')
print(f'dtype: {altitudes.dtype}')

In [ ]:
# Operaciones vectorizadas sobre una Serie
print(altitudes / 1000)          # pasar a km
print(altitudes[altitudes > 2000])  # filtrar

---
## 2. Crear DataFrames

Hay varias formas de construir un DataFrame:

In [ ]:
# Desde un diccionario (la forma más común)
datos = {
    'espacio_natural': ['Doñana', 'Sierra Nevada', 'Tablas de Daimiel',
                        'Teide', 'Ordesa', 'Cabrera', 'Aigüestortes'],
    'ccaa':   ['Andalucía', 'Andalucía', 'Castilla-La Mancha',
               'Canarias', 'Aragón', 'Baleares', 'Cataluña'],
    'sup_ha': [75_000, 86_208, 1_928, 18_990, 15_608, 10_021, 14_119],
    'anio_declaracion': [1969, 1999, 1973, 1954, 1918, 1991, 1955],
    'visitantes_anuales': [400_000, 900_000, 85_000, 4_200_000,
                           600_000, 110_000, 350_000],
    'patrimonio_mundial': [True, True, False, True, False, True, False],
}

parques = pd.DataFrame(datos)
parques

In [ ]:
# Desde una lista de diccionarios (útil al construir fila a fila)
filas = [
    {'isla': 'Tenerife',     'area_km2': 2034, 'poblacion': 920_000},
    {'isla': 'Gran Canaria', 'area_km2': 1560, 'poblacion': 852_000},
    {'isla': 'Lanzarote',    'area_km2':  846, 'poblacion': 155_000},
    {'isla': 'La Palma',     'area_km2':  708, 'poblacion':  83_000},
]
islas = pd.DataFrame(filas)
islas

---
## 3. Explorar datos

Siempre que cargamos un dataset nuevo, lo primero es entender su estructura.

In [ ]:
print(parques.shape)        # (filas, columnas)
print(parques.columns.tolist())
print(parques.dtypes)

In [ ]:
parques.head(3)     # primeras filas

In [ ]:
parques.tail(2)     # últimas filas

In [ ]:
parques.info()      # resumen: tipos, nulos, memoria

In [ ]:
parques.describe()  # estadísticas de columnas numéricas

In [ ]:
# describe también sobre columnas de texto
parques.describe(include='object')

---
## 4. Selección e indexación

Pandas ofrece tres formas principales de acceder a los datos:

| Método | Accede por | Ejemplo |
|--------|-----------|---------|
| `[]`   | columna / filas con slice | `df['col']`, `df[0:3]` |
| `.loc[]` | etiquetas (índice) | `df.loc[0, 'col']` |
| `.iloc[]` | posición entera | `df.iloc[0, 2]` |

In [ ]:
# Seleccionar una columna → Serie
parques['sup_ha']

In [ ]:
# Seleccionar varias columnas → DataFrame
parques[['espacio_natural', 'sup_ha', 'ccaa']]

In [ ]:
# loc: por etiqueta de índice y nombre de columna
parques.loc[0, 'espacio_natural']         # celda concreta

In [ ]:
parques.loc[2:4, ['espacio_natural', 'anio_declaracion']]  # rango de filas, cols seleccionadas

In [ ]:
# iloc: por posición entera
parques.iloc[0]         # primera fila como Serie

In [ ]:
parques.iloc[:3, :3]    # primeras 3 filas, primeras 3 columnas

### Filtros booleanos

La forma más potente y habitual de filtrar filas:

In [ ]:
# Parques con más de 50 000 ha
grandes = parques[parques['sup_ha'] > 50_000]
grandes

In [ ]:
# Parques Patrimonio Mundial en Andalucía (operadores: & | ~)
parques[(parques['patrimonio_mundial'] == True) & (parques['ccaa'] == 'Andalucía')]

In [ ]:
# isin: equivalente a varios OR
parques[parques['ccaa'].isin(['Canarias', 'Baleares'])]

In [ ]:
# query: sintaxis más legible para filtros complejos
parques.query('sup_ha > 10_000 and anio_declaracion < 1970')

---
## 5. Añadir, modificar y eliminar columnas

In [ ]:
# Nueva columna calculada
parques['visitantes_por_ha'] = parques['visitantes_anuales'] / parques['sup_ha']
parques['visitantes_por_ha'] = parques['visitantes_por_ha'].round(2)

parques[['espacio_natural', 'sup_ha', 'visitantes_anuales', 'visitantes_por_ha']]

In [ ]:
# Modificar valores con map / apply
parques['ccaa_corta'] = parques['ccaa'].map({
    'Andalucía': 'AND', 'Canarias': 'CAN', 'Aragón': 'ARA',
    'Baleares': 'BAL', 'Cataluña': 'CAT', 'Castilla-La Mancha': 'CLM'
})
parques[['espacio_natural', 'ccaa', 'ccaa_corta']]

In [ ]:
# apply con función lambda
parques['nombre_upper'] = parques['espacio_natural'].apply(lambda x: x.upper())
parques[['espacio_natural', 'nombre_upper']].head(3)

In [ ]:
# Eliminar columnas
parques = parques.drop(columns=['nombre_upper', 'ccaa_corta'])
parques.columns.tolist()

---
## 6. Datos ausentes

En datos reales siempre hay valores nulos (`NaN` — Not a Number).

In [ ]:
# Introducir algunos NaN artificialmente para practicar
parques_con_nulos = parques.copy()
parques_con_nulos.loc[1, 'visitantes_anuales'] = np.nan
parques_con_nulos.loc[4, 'anio_declaracion']   = np.nan

# Detectar nulos
parques_con_nulos.isna()

In [ ]:
# Resumen de nulos por columna
parques_con_nulos.isna().sum()

In [ ]:
# Eliminar filas con cualquier NaN
parques_con_nulos.dropna()

In [ ]:
# Rellenar NaN con un valor
parques_con_nulos['visitantes_anuales'].fillna(parques_con_nulos['visitantes_anuales'].median())

---
## 7. Agrupaciones y agregaciones

`groupby` es una de las operaciones más potentes de pandas.  
Sigue el patrón **Split → Apply → Combine**:

In [ ]:
# Superficie total por comunidad autónoma
parques.groupby('ccaa')['sup_ha'].sum().sort_values(ascending=False)

In [ ]:
# Varias agregaciones a la vez
parques.groupby('ccaa').agg(
    num_parques    = ('espacio_natural', 'count'),
    sup_total_ha   = ('sup_ha', 'sum'),
    visitantes_max = ('visitantes_anuales', 'max'),
    anio_mas_ant   = ('anio_declaracion', 'min'),
).sort_values('sup_total_ha', ascending=False)

In [ ]:
# value_counts: frecuencia de valores en una columna
parques['ccaa'].value_counts()

---
## 8. Combinar DataFrames

### `pd.concat`: apilar DataFrames

Útil para combinar datasets con las mismas columnas (p.ej. datos de distintos años).

In [ ]:
parques_norte = pd.DataFrame({
    'espacio_natural': ['Picos de Europa', 'Urdaibai'],
    'ccaa': ['Asturias/Cantabria/CyL', 'País Vasco'],
    'sup_ha': [64_660, 22_041],
})

parques_todos = pd.concat([parques[['espacio_natural', 'ccaa', 'sup_ha']],
                            parques_norte], ignore_index=True)
parques_todos

### `pd.merge`: unir por columna clave

Equivalente al JOIN de SQL.

In [ ]:
# Tabla de coordinadores por espacio natural (inventada)
coordinadores = pd.DataFrame({
    'espacio_natural': ['Doñana', 'Teide', 'Sierra Nevada', 'Ordesa', 'Cabrera'],
    'coordinador': ['Ana García', 'Luis Pérez', 'Marta Ruiz', 'Pablo Torres', 'Sofía León'],
    'email': ['ana@csic.es', 'luis@csic.es', 'marta@csic.es', 'pablo@csic.es', 'sofia@csic.es'],
})

# INNER JOIN: solo parques que aparecen en ambas tablas
resultado = pd.merge(parques[['espacio_natural', 'ccaa', 'sup_ha']],
                     coordinadores,
                     on='espacio_natural',
                     how='inner')
resultado

In [ ]:
# LEFT JOIN: todos los parques, NaN donde no hay coordinador
pd.merge(parques[['espacio_natural', 'ccaa', 'sup_ha']],
         coordinadores,
         on='espacio_natural',
         how='left')

---
## 9. Leer y escribir archivos

In [ ]:
# Guardar a CSV
parques.to_csv('parques_nacionales.csv', index=False, encoding='utf-8')
print('CSV guardado')

In [ ]:
# Leer CSV
df = pd.read_csv('parques_nacionales.csv')
df.head(3)

In [ ]:
# Opciones útiles al leer
df2 = pd.read_csv('parques_nacionales.csv',
                  usecols=['espacio_natural', 'sup_ha', 'ccaa'],  # solo estas cols
                  dtype={'sup_ha': float})                         # forzar tipo
df2.head(3)

In [ ]:
# read_csv también puede leer directamente URLs (muy útil para datos abiertos)
# url = 'https://ejemplo.com/datos.csv'
# df = pd.read_csv(url)

# Excel
parques.to_excel('parques_nacionales.xlsx', index=False, sheet_name='Parques')
df_excel = pd.read_excel('parques_nacionales.xlsx')
df_excel.head(3)

In [ ]:
# Limpieza
import os
for f in ['parques_nacionales.csv', 'parques_nacionales.xlsx']:
    if os.path.exists(f):
        os.remove(f)

---
## 10. Visualización rápida con pandas

pandas tiene integración directa con matplotlib para gráficas rápidas y exploratorias.

In [ ]:
import matplotlib.pyplot as plt

# Barras: superficie por parque
ax = parques.set_index('espacio_natural')['sup_ha'].sort_values().plot(
    kind='barh', figsize=(9, 4), color='steelblue', title='Superficie (ha)')
ax.set_xlabel('Hectáreas')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter: superficie vs visitantes
ax = parques.plot(kind='scatter', x='sup_ha', y='visitantes_anuales',
                   figsize=(7, 4), title='Superficie vs Visitantes', color='darkorange')
for _, row in parques.iterrows():
    ax.annotate(row['espacio_natural'].split()[0],
                (row['sup_ha'], row['visitantes_anuales']),
                textcoords='offset points', xytext=(4, 4), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Histograma
parques['visitantes_anuales'].plot(kind='hist', bins=6, edgecolor='white',
                                   figsize=(6, 3), title='Distribución de visitantes')
plt.xlabel('Visitantes anuales')
plt.tight_layout()
plt.show()

---
## 11. DataFrame vs GeoDataFrame: la conexión con GeoPandas

Aquí hacemos el puente entre pandas y lo que veremos el Día 2.  
Usaremos dos datasets reales:

- **California Housing** — el dataset de muestra de Google Colab (`sklearn`), con columnas `latitude` y `longitude`
- **Municipios de Andalucía** — el shapefile `data/13_01_TerminoMunicipal.shp`, cargado como GeoDataFrame

La idea clave: **un GeoDataFrame es un DataFrame con una columna `geometry` extra**.

In [ ]:
# Dataset California Housing — el típico de Google Colab / sklearn
from sklearn.datasets import fetch_california_housing
import pandas as pd

housing_raw = fetch_california_housing(as_frame=True)
housing = housing_raw.frame   # DataFrame normal
housing.head()

In [ ]:
print('Tipo:', type(housing))
print('Shape:', housing.shape)
print('Columnas:', housing.columns.tolist())
housing.describe()

In [ ]:
# Tiene coordenadas geográficas — aunque no hay columna 'geometry'
housing[['Latitude', 'Longitude']].head()

### Cargar el shapefile como GeoDataFrame

In [ ]:
import geopandas as gpd
import os

# Ruta relativa desde la carpeta notebooks/dia_1/
data_path = os.path.join('..', '..', 'data', '13_01_TerminoMunicipal.shp')
municipios = gpd.read_file(data_path)
municipios.head()

In [ ]:
print('Tipo:', type(municipios))
print('Shape:', municipios.shape)
print('Columnas:', municipios.columns.tolist())
print('CRS:', municipios.crs)

### Comparación lado a lado

Las operaciones de pandas funcionan igual en ambos:

In [ ]:
# ── Filtrar ──────────────────────────────────────────────────────────────────
# DataFrame normal
housing_caro = housing[housing['MedHouseVal'] > 4.0]
print(f'Viviendas caras: {len(housing_caro)} de {len(housing)}')

# GeoDataFrame — exactamente igual
mun_sevilla = municipios[municipios['provincia'] == 'Sevilla']
print(f'Municipios de Sevilla: {len(mun_sevilla)} de {len(municipios)}')

In [ ]:
# ── Groupby ───────────────────────────────────────────────────────────────────
# Por habitaciones medias (binned)
housing['hab_grupo'] = pd.cut(housing['AveRooms'], bins=[0,3,5,8,100],
                               labels=['<3', '3-5', '5-8', '>8'])
print(housing.groupby('hab_grupo', observed=True)['MedHouseVal'].mean().round(3))

print()

# Por provincia
print(municipios.groupby('provincia')['area'].sum().sort_values(ascending=False).head())

In [ ]:
# ── Visualización ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# DataFrame: scatter geográfico manual
axes[0].scatter(housing['Longitude'], housing['Latitude'],
                c=housing['MedHouseVal'], cmap='YlOrRd',
                s=0.5, alpha=0.5)
axes[0].set_title('California Housing\n(DataFrame → scatter manual)')
axes[0].set_xlabel('Longitud'); axes[0].set_ylabel('Latitud')

# GeoDataFrame: plot directo (usa la columna geometry)
municipios.plot(column='area', cmap='Blues', ax=axes[1],
                legend=True, legend_kwds={'shrink': 0.7})
axes[1].set_title('Municipios de Andalucía\n(GeoDataFrame → .plot() nativo)')
axes[1].set_axis_off()

plt.tight_layout()
plt.show()

### Crear un GeoDataFrame a partir del DataFrame de Housing

Si tenemos un DataFrame con coordenadas, podemos convertirlo a GeoDataFrame en una línea:

In [ ]:
# Convertir el DataFrame de Housing en GeoDataFrame
housing_gdf = gpd.GeoDataFrame(
    housing,
    geometry=gpd.points_from_xy(housing['Longitude'], housing['Latitude']),
    crs='EPSG:4326'   # WGS84
)

print(type(housing_gdf))
print(housing_gdf[['MedHouseVal', 'Latitude', 'Longitude', 'geometry']].head(3))

In [ ]:
# Ahora ya tenemos operaciones espaciales disponibles
housing_gdf.plot(column='MedHouseVal', cmap='YlOrRd',
                 figsize=(7, 5), markersize=0.5, alpha=0.5,
                 legend=True, legend_kwds={'shrink': 0.7})
import matplotlib.pyplot as plt
plt.title('California Housing — GeoDataFrame')
plt.show()

### Resumen de la comparación

| | `pd.DataFrame` | `gpd.GeoDataFrame` |
|---|---|---|
| Hereda de | `object` | `pd.DataFrame` |
| Columna especial | — | `geometry` |
| CRS | — | `.crs` |
| Filtrar | `df[mask]` | igual |
| Groupby | `df.groupby()` | igual |
| Plot | `df.plot()` matplotlib | `.plot()` con ejes geográficos |
| Operaciones espaciales | — | `.area`, `.buffer()`, `.sjoin()`, `.to_crs()`… |

> **Conclusión:** si sabes pandas, ya sabes el 80% de GeoPandas.  
> El Día 2 se centra en ese 20% restante: geometrías, CRS y operaciones espaciales.

---
## Resumen

| Operación | Método |
|-----------|--------|
| Ver datos | `head()`, `tail()`, `info()`, `describe()` |
| Seleccionar columnas | `df['col']`, `df[['c1','c2']]` |
| Filtrar filas | `df[mask]`, `df.query(...)` |
| Acceder por etiqueta / posición | `loc[]`, `iloc[]` |
| Nueva columna | `df['nueva'] = ...` |
| Eliminar | `drop(columns=[...])` |
| Datos nulos | `isna()`, `dropna()`, `fillna()` |
| Agrupar | `groupby().agg(...)` |
| Combinar | `concat()`, `merge()` |
| I/O | `read_csv()`, `to_csv()`, `read_excel()` |
| Gráfica rápida | `df.plot(kind=...)` |
| → GeoDataFrame | `gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(...))` |

**Mañana (Día 2):** Todo esto + geometrías = GeoPandas.

---
*GeoPython 2026 — CSIC / Estación Biológica de Doñana*